# 01 - Order Line to Order Level Aggregation



## Setup

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)

DATA_PATH = Path('../../../data/raw data/DataCoSupplyChainDataset.csv')
OUTPUT_PATH = Path('../../../data/processed/orders_aggregated.csv')

try:
    df = pd.read_csv(DATA_PATH, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(DATA_PATH, encoding='ISO-8859-1')

print(f"Shape before aggregation: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique orders: {df['Order Id'].nunique():,}")


Shape before aggregation: 180,519 rows x 53 columns
Unique orders: 65,752


## 1. Check which planned feature columns actually vary within an order

For every column we planned to carry forward as a feature, check `nunique()` per `Order Id` group. If the max is 1, the column is genuinely constant within an order and `first` is safe. If not, we need a different aggregation rule (or to flag the column as unreliable at order level).


In [15]:
candidate_columns = [
    'Shipping Mode', 'Order Region', 'Order Country', 'Order City', 'Order State',
    'Customer Segment', 'Type', 'Days for shipment (scheduled)',
    'Category Name', 'Category Id', 'Late_delivery_risk', 'Delivery Status',
]
candidate_columns = [c for c in candidate_columns if c in df.columns]

variability_check = df.groupby('Order Id')[candidate_columns].nunique().max()
variability_check.sort_values(ascending=False)


Category Name                    5
Category Id                      5
Shipping Mode                    1
Order Region                     1
Order City                       1
Order Country                    1
Order State                      1
Customer Segment                 1
Days for shipment (scheduled)    1
Type                             1
Late_delivery_risk               1
Delivery Status                  1
dtype: int64

**What we found:**

**Every planned categorical field is genuinely constant within an order - except two: `Category Name` and `Category Id`, which vary up to 5 times within a single order.** This makes sense once you think about it: a customer can order multiple different products (e.g. camping gear + a water sports item) in one order, and each line item can belong to a different product category, while `Shipping Mode`, `Order Region`, `Order Country`, etc. are genuinely order level properties (one shipment, one destination) that never vary across an order's lines.

**This is a real problem with the aggregation rules defined in Section 2 below - they currently use `first` for `Category Name`/`Category Id`, which silently keeps only one category out of up to 5 and discards the rest.** For an order containing camping gear and a water sports item, `first` would arbitrarily record it as just "Camping & Hiking" and lose the fact that it was a mixed category order entirely. That's not a safe simplification - it's quietly throwing away real information, and worse, it's not even *consistent* information (which category ends up as "first" depends on row order in the raw file, not anything meaningful).

**This needs to be fixed before moving to Notebook 02, not just documented.** Two reasonable fixes, and the second one is the better choice here:
1. Use the *mode* (most frequent category in the order) instead of `first` - better than arbitrary, but still discards real multi category information.
2. **Recommended:** keep `Category Name`/`Category Id` as a *descriptive* field (mode, for reference/reporting) but add a new feature - `n_distinct_categories` - capturing how many different categories appeared in the order. This preserves the real signal (a multi category order is a genuinely different kind of order than a single item one, and might carry different delay risk) instead of silently erasing it.

**Action needed:** update the aggregation dictionary in Section 2 to use mode for `Category Name`/`Category Id` and add the `n_distinct_categories` feature before re running and saving.


## 2. Define the Aggregation Rule per Column

After checking how each feature varies within an order, we need to decide how to combine multiple item level rows into **one row per order**.

The aggregation rules are:

| Column / Feature Type | Aggregation Rule | Reason |
|---|---|---|
| **Sales** | `sum` | Calculates the total revenue for the entire order. |
| **Order Item Quantity** | `sum` | Calculates the total number of items/units in the order. |
| **Benefit** | `sum` | Calculates the total benefit/profit generated by the order. |
| **Order level categorical features** | `first` | These features are constant within an order, so taking the first value is safe. |
| **Target** | `first` | The target is constant within an order, so we retain the single order-level value. |

### Example

If an order contains three items:

| Order Id | Sales | Order Item Quantity | Benefit | Customer City |
|---|---:|---:|---:|---|
| 1001 | 500 | 1 | 50 | Colombo |
| 1001 | 200 | 2 | 20 | Colombo |
| 1001 | 100 | 1 | 10 | Colombo |

After aggregation, the order becomes:

| Order Id | Sales | Order Item Quantity | Benefit | Customer City |
|---|---:|---:|---:|---|
| 1001 | 800 | 4 | 80 | Colombo |

This means:

- **Sales:** `500 + 200 + 100 = 800`
- **Order Item Quantity:** `1 + 2 + 1 = 4`
- **Benefit:** `50 + 20 + 10 = 80`
- **Customer City:** `Colombo` is constant, so we keep the first value.

### Decision Log Addition

The original `DECISION_LOG.md` entry dated **2026-09-17** explicitly specified:

- `sum` for **Sales**
- `sum` for **Order Item Quantity**
- `first` for **order level categorical features**
- `first` for the **target**

We additionally specify **`sum` for Benefit** at the order level.

This addition is logged because the original decision did not explicitly mention Benefit. Summing Benefit gives **total order profit/benefit**, which is consistent with treating Sales as **total order revenue**.

### Overall Objective

The purpose of these aggregation rules is to transform the dataset from:

**multiple item level rows per order → one row representing the complete order.**


In [16]:
def mode_or_first(series):
    modes = series.mode()
    return modes.iloc[0] if not modes.empty else series.iloc[0]

agg_rules = {
    # Target and outcome (post-outcome fields kept only for the audit trail, not as features)
    'Late_delivery_risk': 'first',
    'Delivery Status': 'first',

    # Order-time-safe categoricals (constant within an order per the check above)
    'Shipping Mode': 'first',
    'Order Region': 'first',
    'Order Country': 'first',
    'Order City': 'first',
    'Order State': 'first',
    'Order Zipcode': 'first',
    'Customer Segment': 'first',
    'Type': 'first',
    'Days for shipment (scheduled)': 'first',

    # Category — FIXED: these vary within an order (found in Section 1), so use mode
    # instead of first, and add a separate n_distinct_categories feature below.
    'Category Name': mode_or_first,
    'Category Id': mode_or_first,

    # Financial / value fields — summed to represent the whole order
    'Sales': 'sum',
    'Order Item Quantity': 'sum',
    'Benefit per order': 'sum',

    # Temporal
    'order date (DateOrders)': 'first',
}
agg_rules = {k: v for k, v in agg_rules.items() if k in df.columns}

orders_df = df.groupby('Order Id').agg(agg_rules).reset_index()

# New engineered features
orders_df['n_line_items'] = df.groupby('Order Id').size().values
orders_df['n_distinct_categories'] = df.groupby('Order Id')['Category Name'].nunique().values

print(f"Shape after aggregation: {orders_df.shape[0]:,} rows x {orders_df.shape[1]} columns")
print(f"\nOrders with more than 1 distinct category: {(orders_df['n_distinct_categories'] > 1).sum()} ({(orders_df['n_distinct_categories'] > 1).mean()*100:.2f}%)")
orders_df.head()


Shape after aggregation: 65,752 rows x 20 columns

Orders with more than 1 distinct category: 44563 (67.77%)


,Order Id,Late_delivery_risk,Delivery Status,Shipping Mode,Order Region,Order Country,Order City,Order State,Order Zipcode,Customer Segment,Type,Days for shipment (scheduled),Category Name,Category Id,Sales,Order Item Quantity,Benefit per order,order date (DateOrders),n_line_items,n_distinct_categories
0,1,0,Advance shipping,Standard Class,Central America,México,Mexico City,Distrito Federal,NaN,Consumer,CASH,4,Camping & Hiking,43,299.980011,1,88.790001,1/1/2015 0:00,1,1
1,2,0,Advance shipping,Standard Class,South America,Colombia,Dos Quebradas,Risaralda,NaN,Consumer,PAYMENT,4,Men's Footwear,18,579.980011,7,195.900002,1/1/2015 0:21,3,3
2,4,1,Late delivery,Standard Class,South America,Colombia,Dos Quebradas,Risaralda,NaN,Home Office,CASH,4,Accessories,17,699.850010,14,124.090000,1/1/2015 1:03,4,4
3,5,1,Late delivery,Standard Class,South America,Colombia,Dos Quebradas,Risaralda,NaN,Consumer,DEBIT,4,Camping & Hiking,43,1129.860039,10,390.089995,1/1/2015 1:24,5,4
4,7,1,Late delivery,Second Class,South America,Brasil,São Paulo,São Paulo,NaN,Consumer,DEBIT,2,Camping & Hiking,41,579.920013,7,203.929998,1/1/2015 2:06,3,3


## 3. Sanity checks before saving

Confirm the aggregation didn't silently break anything - row count should exactly match unique order count, and the target should have no missing values.


In [17]:
assert orders_df.shape[0] == df['Order Id'].nunique(), "Row count mismatch after aggregation!"
assert orders_df['Late_delivery_risk'].isnull().sum() == 0, "Target has missing values after aggregation!"

print("Sanity checks passed.")
print(f"\nClass balance after aggregation:")
print(orders_df['Late_delivery_risk'].value_counts(normalize=True).round(4) * 100)


Sanity checks passed.

Class balance after aggregation:
Late_delivery_risk
1    54.82
0    45.18
Name: proportion, dtype: float64


**What we found:**

**54.82% late / 45.18% not-late after aggregation - a near exact match to the 54.83%/45.17% split found at line item level in EDA.** This confirms the aggregation didn't introduce any bias or bug in how the target was carried forward - `Late_delivery_risk` collapsed cleanly via `first`, exactly as expected given the EDA finding that this field is identical across all line items within an order.




## 4. Save the aggregated dataset

In [18]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
orders_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH.resolve()}")


Saved to: C:\Users\Ewis\Documents\Machine_learning_project\Heads-Up_IT3091\data\processed\orders_aggregated.csv


**`DECISION_LOG.md`:** final aggregation rule confirmed per column (including the addition of summing `Benefit per order` and the new `n_line_items` feature), based on the variability. 
